In [ ]:
import pandas as pd
import numpy as np
import math as mt
import os
import matplotlib.pyplot as plt
import sys
import traceback
import contextlib
import io

sys.path.append(os.path.abspath('..'))  # REDLAB_UQ.py is in the project root

try:
    from REDLAB_UQ import SurrogateModel
    import REDLAB_UQ
    import importlib

    importlib.reload(REDLAB_UQ)
    from REDLAB_UQ import SurrogateModel
    print("REDLAB_UQ module reloaded.")
except ImportError as e:
    print(f"REDLAB_UQ import failed: {e}")
    print("Check that REDLAB_UQ.py is in the project root folder.")
except Exception as e:
    print(f"Unexpected import error: {e}")

model = SurrogateModel()


## 1. Load data


In [ ]:
# base_values = [
#     26.894765993870866,
#     26.78963336562726,
#     60.5466610514048,
#     7.870646571918311,
#     70.00000019778525,
#     304.9999999981768,
# ]  # optimal, sample 13

base_values = [36, 36, 70, 5, 70, 305]  # Initial

# Use 1-based sample numbers here if a completed output row must be excluded.
#skipped_sample_indices = [] # optimal
skipped_sample_indices = [73] # initial

input_path = "../initial/input_samples_zz.csv"
output_path = "../initial/sample_1.csv" # initial design

df_inputs_z = pd.read_csv(input_path)
df_values = pd.read_csv(output_path)

input_samples_all = df_inputs_z.to_numpy(dtype=float)
output_samples_all = df_values.to_numpy(dtype=float)

input_means = np.asarray(base_values, dtype=float)
target_values = list(range(output_samples_all.shape[1]))
target_labels = list(df_values.columns)

n_output_samples = output_samples_all.shape[0]
skipped_indices_0based = {idx - 1 for idx in skipped_sample_indices}
valid_indices = np.array(
    [
        idx
        for idx in range(n_output_samples)
        if idx not in skipped_indices_0based
    ],
    dtype=int,
)

input_samples_coeff = input_samples_all[valid_indices, :]
output_samples_coeff = output_samples_all[valid_indices, :][:, target_values]

data_overview_df = pd.DataFrame(
    {
        "Item": [
            "Input samples for whitening",
            "Available output samples",
            "Used output samples",
            "Skipped output samples",
            "Input variables",
            "Outputs",
        ],
        "Value": [
            input_samples_all.shape[0],
            n_output_samples,
            output_samples_coeff.shape[0],
            len(skipped_sample_indices),
            input_samples_coeff.shape[1],
            output_samples_coeff.shape[1],
        ],
    }
)

display(data_overview_df)


## 2. Convergence settings


In [ ]:
N = input_samples_coeff.shape[1]
S = 1
m_values = list(range(1, 5))
regression_method = "SLS"
rcond = 1e-12

# The whitening transformation uses all 1,000,000 input samples to represent the input distribution.
whitening_sample_count = min(1_000_000, input_samples_all.shape[0])
input_samples_whitening = input_samples_all[:whitening_sample_count, :]

print(f"N={N}, S={S}, m_values={m_values}, regression={regression_method}")
print(f"Training output samples: {output_samples_coeff.shape[0]}")
print(f"Whitening input samples: {input_samples_whitening.shape[0]}")


## 3. Fit DD-GPCE for m = 1..4


In [ ]:
def fit_dd_gpce_for_order(m):
    basis_num = model.calculate_basis_num(N, m, S)
    n_train = output_samples_coeff.shape[0]

    if n_train < basis_num:
        raise ValueError(
            f"m={m} has {basis_num} basis terms, but only {n_train} output samples are available."
        )

    train_basis_whitening = model.P_grlex_n_DD_fast(input_samples_whitening, m, S)
    with contextlib.redirect_stdout(io.StringIO()):
        W_DD = model.whitening_transform(
            train_basis_whitening,
            input_samples_whitening.shape[0],
        )

    train_basis_coeff = model.P_grlex_n_DD_fast(input_samples_coeff, m, S)
    A_T = W_DD @ train_basis_coeff
    A = A_T.T
    B = output_samples_coeff

    coefficients, residuals, rank, singular_values = np.linalg.lstsq(
        A,
        B,
        rcond=rcond,
    )

    means = coefficients[0, :]
    variances = np.sum(coefficients[1:, :] ** 2, axis=0)
    stds = np.sqrt(np.maximum(variances, 0.0))

    return {
        "m": m,
        "basis_num": basis_num,
        "n_train": n_train,
        "rank": int(rank),
        "condition_number": float(np.linalg.cond(A)),
        "coefficients": coefficients,
        "means": means,
        "stds": stds,
    }


order_results = [fit_dd_gpce_for_order(m) for m in m_values]


## 4. Mean/std convergence from m-1 to m


In [ ]:
convergence_rows = []
eps = 1e-12

previous_result = None
for result in order_results:
    m = result["m"]

    for output_index, output_name in enumerate(target_labels):
        mean_value = result["means"][output_index]
        std_value = result["stds"][output_index]

        if previous_result is None:
            mean_change_pct = np.nan
            std_change_pct = np.nan
        else:
            previous_mean = previous_result["means"][output_index]
            previous_std = previous_result["stds"][output_index]
            mean_change_pct = (
                abs(mean_value - previous_mean)
                / max(abs(previous_mean), eps)
                * 100
            )
            std_change_pct = (
                abs(std_value - previous_std)
                / max(abs(previous_std), eps)
                * 100
            )

        convergence_rows.append(
            {
                "m": m,
                "Output": output_name,
                "Basis num": result["basis_num"],
                "Samples": result["n_train"],
                "rank(A)": result["rank"],
                "cond(A)": result["condition_number"],
                "Mean": mean_value,
                "Std": std_value,
                "Mean change vs m-1 (%)": mean_change_pct,
                "Std change vs m-1 (%)": std_change_pct,
            }
        )

    previous_result = result

convergence_detail_df = pd.DataFrame(convergence_rows)

convergence_summary_df = (
    convergence_detail_df
    .groupby("m", as_index=False)
    .agg(
        **{
            "Basis num": ("Basis num", "first"),
            "Samples": ("Samples", "first"),
            "rank(A)": ("rank(A)", "first"),
            "Max mean change (%)": ("Mean change vs m-1 (%)", "max"),
            "Avg mean change (%)": ("Mean change vs m-1 (%)", "mean"),
            "Max std change (%)": ("Std change vs m-1 (%)", "max"),
            "Avg std change (%)": ("Std change vs m-1 (%)", "mean"),
        }
    )
)
convergence_summary_df["Max combined change (%)"] = convergence_summary_df[
    ["Max mean change (%)", "Max std change (%)"]
].max(axis=1)

display(convergence_summary_df.round(6))


In [ ]:
display(convergence_detail_df.round(6))


In [ ]:
plot_df = convergence_summary_df.dropna(subset=["Max combined change (%)"])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    plot_df["m"],
    plot_df["Max mean change (%)"],
    marker="o",
    linewidth=2,
    label="Max mean change",
)
ax.plot(
    plot_df["m"],
    plot_df["Max std change (%)"],
    marker="s",
    linewidth=2,
    label="Max std change",
)
ax.set_xlabel("Polynomial order m")
ax.set_ylabel("Change from previous order (%)")
ax.set_title("DD-GPCE moment convergence by polynomial order")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Interpretation

- For `m=1`, the change is `NaN` because there is no lower order.
- `Mean change vs m-1 (%)` and `Std change vs m-1 (%)` in the row `m=n` are the changes relative to `m=n-1` for the same output.
- Convergence is usually judged by whether `Max combined change (%)` in the summary table becomes sufficiently small.
- The same output samples are used for all m, so that only the effect of the polynomial order is compared.
- `S=1` keeps the DD-GPCE interaction order used elsewhere; to increase the interaction order, adjust `S` separately.


## 5. Direct 100 vs DD-GPCE distribution comparison

The cells below build the DD-GPCE distribution with the selected `distribution_m` and `distribution_S` and compare it with the distribution of the 100 high-fidelity samples; the convergence study above is unchanged.


In [ ]:
from pathlib import Path
from scipy import stats

# Distribution comparison settings.
# By default, the same setting as the last m of the convergence study above.
distribution_m = m_values[-1]
distribution_S = S

# Number of input samples used to build the DD-GPCE surrogate distribution.
# Set to 1_000_000 to use the whole input pool.
gpce_distribution_sample_count = min(1_000_000, input_samples_all.shape[0])

n_bins = 40
figure_output_path = Path(
    "dd_gpce_distribution_vs_direct_100.png"
)

distribution_basis_num = model.calculate_basis_num(N, distribution_m, distribution_S)
if output_samples_coeff.shape[0] < distribution_basis_num:
    print(
        f"Warning: direct samples ({output_samples_coeff.shape[0]}) < "
        f"basis terms ({distribution_basis_num}). "
        "np.linalg.lstsq will return a minimum-norm solution."
    )

distribution_settings_df = pd.DataFrame(
    {
        "Setting": [
            "distribution_m",
            "distribution_S",
            "Basis num",
            "Direct simulation samples",
            "DD-GPCE distribution samples",
        ],
        "Value": [
            distribution_m,
            distribution_S,
            distribution_basis_num,
            output_samples_coeff.shape[0],
            gpce_distribution_sample_count,
        ],
    }
)

display(distribution_settings_df)


In [ ]:
def build_whitening_transform_safe(monomial_matrix, sample_count, jitter=1e-12):
    """Whitening matrix for REDLAB basis matrices shaped (basis, samples)."""
    gram = (monomial_matrix @ monomial_matrix.T) / sample_count
    gram = 0.5 * (gram + gram.T)

    try:
        q = np.linalg.cholesky(gram)
        return np.linalg.inv(q), "cholesky"
    except np.linalg.LinAlgError:
        eigvals, eigvecs = np.linalg.eigh(gram)
        eig_floor = jitter * max(float(np.max(eigvals)), 1.0)
        eigvals = np.clip(eigvals, eig_floor, None)
        return np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T, "eigh fallback"


def build_design_matrix_safe(X_z, whitening_matrix, m, S):
    basis = model.P_grlex_n_DD_fast(X_z, m, S)
    return (whitening_matrix @ basis).T


X_distribution_z = input_samples_all[:gpce_distribution_sample_count, :]

basis_for_whitening = model.P_grlex_n_DD_fast(
    X_distribution_z,
    distribution_m,
    distribution_S,
)
W_distribution, whitening_method = build_whitening_transform_safe(
    basis_for_whitening,
    X_distribution_z.shape[0],
)

A_direct_distribution = build_design_matrix_safe(
    input_samples_coeff,
    W_distribution,
    distribution_m,
    distribution_S,
)

distribution_coefficients, residuals, rank, singular_values = np.linalg.lstsq(
    A_direct_distribution,
    output_samples_coeff,
    rcond=rcond,
)

A_gpce_distribution = build_design_matrix_safe(
    X_distribution_z,
    W_distribution,
    distribution_m,
    distribution_S,
)
Y_gpce_distribution = A_gpce_distribution @ distribution_coefficients
Y_gpce_on_direct_inputs = A_direct_distribution @ distribution_coefficients

distribution_model_info_df = pd.DataFrame(
    {
        "Item": [
            "m",
            "S",
            "Basis num",
            "Direct samples",
            "DD-GPCE distribution samples",
            "rank(A)",
            "cond(A)",
            "Whitening",
        ],
        "Value": [
            distribution_m,
            distribution_S,
            distribution_basis_num,
            output_samples_coeff.shape[0],
            X_distribution_z.shape[0],
            int(rank),
            float(np.linalg.cond(A_direct_distribution)),
            whitening_method,
        ],
    }
)

display(distribution_model_info_df)


In [ ]:
distribution_rows = []
eps = 1e-12

for output_index, output_name in enumerate(target_labels):
    direct_data = output_samples_coeff[:, output_index]
    gpce_data = Y_gpce_distribution[:, output_index]

    direct_mean = np.mean(direct_data)
    gpce_mean = np.mean(gpce_data)
    direct_std = np.std(direct_data)
    gpce_std = np.std(gpce_data)
    ks_statistic, ks_pvalue = stats.ks_2samp(direct_data, gpce_data, method="auto")

    distribution_rows.append(
        {
            "Output": output_name,
            "Direct N": direct_data.shape[0],
            "DD-GPCE N": gpce_data.shape[0],
            "Direct mean": direct_mean,
            "DD-GPCE mean": gpce_mean,
            "Mean error (%)": abs(gpce_mean - direct_mean) / max(abs(direct_mean), eps) * 100,
            "Direct std": direct_std,
            "DD-GPCE std": gpce_std,
            "Std error (%)": abs(gpce_std - direct_std) / max(abs(direct_std), eps) * 100,
            "KS statistic": ks_statistic,
            "KS p-value": ks_pvalue,
        }
    )

distribution_comparison_df = pd.DataFrame(distribution_rows)
display(distribution_comparison_df.round(8))


In [ ]:
n_outputs = len(target_labels)
n_cols = min(2, n_outputs)
n_rows = int(np.ceil(n_outputs / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(n_cols * 7.0, n_rows * 5.0),
    squeeze=False,
)
axes = axes.flatten()

for output_index, output_name in enumerate(target_labels):
    ax = axes[output_index]
    direct_data = output_samples_coeff[:, output_index]
    gpce_data = Y_gpce_distribution[:, output_index]

    data_min = min(np.min(direct_data), np.min(gpce_data))
    data_max = max(np.max(direct_data), np.max(gpce_data))
    if np.isclose(data_min, data_max):
        delta = max(abs(data_min), 1.0) * 0.05
        data_min -= delta
        data_max += delta

    bins = np.linspace(data_min, data_max, n_bins + 1)

    ax.hist(
        gpce_data,
        bins=bins,
        density=True,
        alpha=0.72,
        color="#ff7f0e",
        edgecolor="#ff7f0e",
        linewidth=0.5,
        label=f"DD-GPCE (n={gpce_data.shape[0]:,})",
    )
    ax.hist(
        direct_data,
        bins=bins,
        density=True,
        histtype="step",
        color="#1f77b4",
        linewidth=2.4,
        label=f"Direct 100 (n={direct_data.shape[0]})",
    )

    direct_mean = np.mean(direct_data)
    gpce_mean = np.mean(gpce_data)
    direct_std = np.std(direct_data)
    gpce_std = np.std(gpce_data)
    ks_pvalue = distribution_comparison_df.loc[
        distribution_comparison_df["Output"] == output_name,
        "KS p-value",
    ].iloc[0]

    ax.axvline(direct_mean, color="#174ea6", linestyle="-", linewidth=2.0)
    ax.axvline(gpce_mean, color="#c45a00", linestyle="--", linewidth=2.0)
    ax.set_title(
        f"{output_name}\n"
        f"Direct mu={direct_mean:.5g}, sigma={direct_std:.5g} | "
        f"GPCE mu={gpce_mean:.5g}, sigma={gpce_std:.5g}\n"
        f"KS p-value={ks_pvalue:.4g}",
        fontsize=11,
    )
    ax.set_xlabel("Output value")
    ax.set_ylabel("Density")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

for empty_index in range(n_outputs, len(axes)):
    axes[empty_index].set_visible(False)

fig.suptitle(
    f"Direct 100 samples vs DD-GPCE distribution (m={distribution_m}, S={distribution_S})",
    fontsize=15,
    fontweight="bold",
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(figure_output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure: {figure_output_path}")


## Distribution comparison notes

- This section is an additional check for the selected `distribution_m` and `distribution_S`; the convergence study above is unchanged.
- `Direct 100` denotes the 100 output samples obtained from the high-fidelity analyses.
- `DD-GPCE` is the distribution obtained by fitting the coefficients to the same 100 output samples and evaluating the surrogate at `gpce_distribution_sample_count` input samples.
- The KS p-value is sensitive to the difference in sample sizes and should be interpreted as a supplementary indicator together with the mean and standard-deviation errors.
